In [ ]:
# this notebook should show the main steps to incorporate BASiL in a gammapy analysis
# and compute main auxiliary stuff
# maybe a separate notebook for 3D

# I can try doing a "BASiL" analysis by faking values and distribution of GAMMANNESS 
# using the DR1 data (from gammapy)

In [ ]:
from gammapy.data import DataStore

In [ ]:
# Crab example

obs_ids = [23523, 23526, 23559, 23592]
datastore_std = DataStore.from_dir("./data/std_zeta/")
observations_std_prod5 = datastore_std.get_observations(obs_ids, required_irf=["aeff", "edisp", "psf"])

pos = observations_std_prod5[0].meta.target.position
ra_obj = pos.icrs.ra.value
dec_obj = pos.icrs.dec.value
name_obj = 'Crab'
target = SkyCoord(ra_obj, dec_obj, frame='icrs', unit='deg')

e_reco = np.logspace(-1, 2, 13) * u.TeV 
e_true = np.logspace(-1, 2, 49) * u.TeV 

energy_axis = MapAxis.from_edges(e_reco, unit='TeV', name='energy', interp='log')
energy_axis_true = MapAxis.from_edges(e_true, unit='TeV', name="energy_true", interp='log')

offset_max = 2*u.deg

target_position = pos
on_region_radius = Angle("0.1118034 deg")
on_region = CircleSkyRegion(center=target_position, radius=on_region_radius)

geom = RegionGeom.create(region=on_region, axes=[energy_axis])
dataset_empty = SpectrumDataset.create(geom=geom, energy_axis_true=energy_axis_true)

dataset_maker = SpectrumDatasetMaker(
    containment_correction=True, selection=["counts", "exposure", "edisp"]
)
bkg_maker = ReflectedRegionsBackgroundMaker()
safe_mask_maker = SafeMaskMaker(
                       methods=["offset-max", 'aeff-max', 'edisp-bias'],
                       offset_max=offset_max, bias_percent=10, aeff_percent=15)

datasets_prod5 = Datasets()
for obs_id, observation in zip(obs_ids, observations_std_prod5):
    dataset = dataset_maker.run(dataset_empty.copy(name=str(obs_id)), observation)
    dataset_on_off = bkg_maker.run(dataset, observation)
    dataset_on_off = safe_mask_maker.run(dataset_on_off, observation)
    datasets_prod5.append(dataset_on_off)
    print('Ethreshold bin: {:}'.format(np.argwhere(dataset_on_off.mask_safe.data == True)[0][0]))

stacked_prod5 = datasets_prod5.stack_reduce()
#dataset_on_off.write("./Sources/MC/nozeta_cuts/"+obs_id_+"/indx_"+indx_label+"/amp_"+amp_label+"/spec_dataset/dataset_standard_run"+obs_id_+"_random-seed_offset0-5_norm30_pl_indx"+indx_label+"_amp"+amp_label+"_zetacuts.fits.gz", overwrite=True)

In [ ]:
datasets_basil = Datasets()
for m in range(len(obs_ids)):
    dataset_on_off = datasets_prebasil[m]
    run = obs_ids[m]
        
    events = EventList.read('/home/matheus/Documents/BASiL/data/hessdata/FITS/noBDT-fullEnclosure-oldlookup-olddst/out/run023400-023599/run0'+str(run)+'/hess_events_0'+str(run)+'.fits.gz')
    
    # Initialize BASiL Spectrum Dataset On/Off
    dataset_onoff_basil = SpectrumDatasetOnOffBASiL()
    # Include information from standard dataset
    dataset_onoff_basil.counts = dataset_on_off.counts
    dataset_onoff_basil.counts_off = dataset_on_off.counts_off
    dataset_onoff_basil.exposure = dataset_on_off.exposure
    dataset_onoff_basil.acceptance = dataset_on_off.acceptance
    dataset_onoff_basil.acceptance_off = dataset_on_off.acceptance_off
    dataset_onoff_basil.gti = dataset_on_off.gti
    dataset_onoff_basil.mask_fit = dataset_on_off.mask_fit
    dataset_onoff_basil.psf = dataset_on_off.psf
    dataset_onoff_basil.edisp = dataset_on_off.edisp
    #dataset_onoff_basil.models = dataset_on_off.models
    dataset_onoff_basil.mask_safe = dataset_on_off.mask_safe
    dataset_onoff_basil.meta_table = dataset_on_off.meta_table

    # Values
    energy = dataset_onoff_basil.counts.geom.axes["energy"].center.value
    Non = dataset_onoff_basil.counts.data
    
    # Identify the events inside the On region
    # Construct the res list from EventList
    energy_axis = dataset_onoff_basil.counts.geom.axes["energy"]
    en_bins = energy_axis.edges.value
    on_region_radius = Angle("0.1118034 deg") # HESS default is 0.118
    theta = events.radec.separation(pos)
    on_mask = np.argwhere(theta.value <= on_region_radius.value)

    event_en = events.table['ENERGY'][on_mask].value.flatten()
    events_zeta = 1 - events.table['GAMMANESS'][on_mask].value.flatten() 
    res = []
    for i in range(len(Non)):
        mask_en = np.argwhere((event_en > en_bins[i]) & (event_en < en_bins[i+1])).flatten()
        if len(mask_en) == 0:
            res.append([])
        else:
            res.append(events_zeta[mask_en].flatten())

    # Save res
    #with open('./Sources/Crab/new/'+str(run)+'/res_event-map_run'+str(run)+'_full.pkl', 'wb') as h:
    #    pickle.dump(res, h, pickle.HIGHEST_PROTOCOL)
    
    maxlength = 0
    for i in range(len(res)):
        if np.size(res[i]) > maxlength:
            maxlength = np.size(res[i])
        
    zetabdt_axis = MapAxis.from_bounds(1, maxlength, nbin=maxlength, name="zetabdt", interp="lin", node_type='center')
    var_Non = RegionNDMap.create(dataset_onoff_basil.counts.geom.region, axes=[zetabdt_axis, energy_axis])
    dataset_onoff_basil.var_Non = var_Non

    # Inserting the data values in var_Non
    for i in range(len(energy_axis.center)):
        for j in range(len(zetabdt_axis.center)):
            if j >= len(res[i]):
                dataset_onoff_basil.var_Non.data[i][j] = -1
            else:
                dataset_onoff_basil.var_Non.data[i][j] = res[i][j]
            
    #dataset_onoff_basil.var_Non.write('./Sources/Crab/new/'+str(run)+'/varNon_event-map_run'+str(run)+'_full.fits', overwrite=True)

    datasets_basil.append(dataset_onoff_basil)

In [ ]:
for i in range(len(datasets_basil)):
    varNon = [datasets_basil[i].var_Non.data]
    label = 'crab/new/'+str(obs_ids[i])
    
    binning = list(np.linspace(0,0.83,25)) + list(np.linspace(0.83,1,30)[1:])
    binning_dict = {'zetabdt': [10, binning]}
    # Distribution settings
    phase='1b'
    file_path = '/home/matheus/Documents/BASiL/dist/'
    zenith='45'
    azimuth='north'
    offset_label='0.5'
    variable=['zetabdt']
    workdir = "/home/matheus/Documents/BASiL/dist/factors/"+label+"/"
    os.makedirs(workdir, exist_ok=True)
    os.makedirs(workdir+"/likelihoods/", exist_ok=True)

    ## Compute the likelihood factors
    #basil_class = basil_factors(n_on=datasets_basil[i].counts.data, n_off=datasets_basil[i].counts_off.data, alpha=datasets_basil[i].alpha.data, var_Non=varNon, energy=energy_axis.center.value)

    #basil_class.run(variable, binning_dict, file_path, str(zenith), azimuth, offset_label, phase, label, store_likelihoods=True)
    
    dict_logprod = {}
    for j in range(len(en_bins)-1):
        en_test = datasets_basil[i].counts.geom.axes["energy"].center.value[j]
        with open('/home/matheus/Documents/BASiL/dist/factors/'+label+'/prod_bin'+str(en_test)+'.pkl', 'rb') as file:
            logprod = pickle.load(file)
        dict_logprod[str(en_test)] = logprod

    datasets_basil[i].dict_logprod = dict_logprod
    